# LLM Bench: notebook edition

A second entry point next to the LLM Bench Console web app. Both use the same backend:

```
Notebook / console ──INSERT──▶ BENCH_RUNS ──EXECUTE TASK──▶ BENCH_RUNNER ──▶ PROCESS_QUEUE ──▶ EXECUTE_RUN ──▶ bench job
```

- **The run executes in the background, inside the task.** This notebook only submits, monitors and analyses. Closing the notebook or losing the session does not stop the run; reopen it and re-run section 4.
- Prerequisites: the backend is installed with `console-backend/install.sh`, and the target service is registered in `BENCH_TARGETS`.
- The current role needs read/write on the bench schema and the `EXECUTE TASK` privilege.
- Runtime: run this notebook on a **separate compute pool** (for example a CPU_X64_XS pool such as `BENCH_NB_POOL`) that the current role has USAGE on. **Do not use the bench client pool (`BENCH_CPU_POOL`)**: the notebook would occupy the node the bench job needs, and the job would stay PENDING.

| Section | Content |
|---|---|
| 1 | Parameters (the only cell you need to edit) |
| 2 | Pre-flight checks: target, service status, client pool occupancy |
| 3 | Submit a run |
| 4 | Monitor progress (re-runnable) / cancel |
| 5 | Results and SLA verdicts |
| 6 | Charts: compare runs |
| 7 | Cold start |
| 8 | Diagnostics: vLLM startup log |
| 9 | Clean up |

## 1. Parameters

Edit only this cell. `RUN_ID` may contain letters, digits and underscores (max 40 characters) and must not already exist.

In [ ]:
import re, json, time, datetime as dt
from snowflake.snowpark.context import get_active_session
session = get_active_session()

BENCH_SCHEMA = "QWEN_MULTI_DB.QWEN_MULTI_SCH"   # schema where the backend is installed
TARGET_NAME  = "qwen-demo-account"                    # must exist in BENCH_TARGETS
TESTS        = ["baseline", "ramp", "shape"]     # any of: baseline, ramp, shape, coldstart, replica
LEVELS       = "1 2 4 8 16 32"                   # concurrency steps, space separated
MIN_PROMPTS  = 8                                 # minimum requests per step (20 recommended for formal runs)
SLA          = {"ttft_p95_s": 2, "e2e_p95_s": 30, "min_tok_s": 15}
CONFIRM_DISRUPTIVE = False                       # must be True for coldstart / replica (they restart or scale the service)
RUN_ID       = "nb_" + dt.datetime.utcnow().strftime("%m%d_%H%M")
NOTE         = "Submitted from notebook"

COMPARE_RUNS = ["r1", "r2_cudagraph"]            # runs to compare in sections 5-6 (RUN_ID is added automatically)
EVENT_TABLE  = ""   # account event table (DB.SCHEMA.TABLE), used in section 8; leave empty to skip

# --- validation (these values end up in SQL) ---
ID_RE = re.compile(r"^[A-Za-z0-9_]{1,40}$")
assert re.fullmatch(r"[A-Za-z0-9_$]+\.[A-Za-z0-9_$]+", BENCH_SCHEMA), "BENCH_SCHEMA must be DB.SCHEMA"
assert ID_RE.match(RUN_ID), "RUN_ID: letters, digits, underscore, max 40"
assert all(ID_RE.match(r) for r in COMPARE_RUNS), "COMPARE_RUNS contains an invalid id"
assert set(TESTS) <= {"baseline", "ramp", "shape", "coldstart", "replica"}, "unknown test name"
assert re.fullmatch(r"[0-9]{1,3}( [0-9]{1,3}){0,11}", LEVELS), "LEVELS must look like '1 2 4 8'"
assert EVENT_TABLE == "" or re.fullmatch(r"[A-Za-z0-9_$.]+", EVENT_TABLE), "EVENT_TABLE must be a plain identifier"

session.use_schema(BENCH_SCHEMA)
RUNS = list(dict.fromkeys(COMPARE_RUNS + [RUN_ID]))
RUN_LIST_SQL = ", ".join(f"'{r}'" for r in RUNS)     # safe: every id passed the regex above
print("schema:", session.get_current_database(), session.get_current_schema(), "| role:", session.get_current_role())
print("RUN_ID:", RUN_ID, "| compare:", RUNS)

## 2. Pre-flight checks

### 2.1 Registered targets

In [ ]:
%%sql -r targets
SELECT TARGET_NAME, SERVICE_FQN, SERVER_CONTAINER, BASE_URL, MODEL, CLIENT_POOL, ALLOW_DISRUPTIVE, DESCRIPTION
FROM {{BENCH_SCHEMA}}.BENCH_TARGETS
ORDER BY TARGET_NAME

### 2.2 Target service status and client pool occupancy

If another service (for example someone's notebook) already runs on the client pool, the bench job may stay PENDING with `insufficient memory resources`. Raise that pool's `MAX_NODES` or use a dedicated pool.

In [ ]:
t = session.sql("SELECT * FROM BENCH_TARGETS WHERE TARGET_NAME = ?", params=[TARGET_NAME]).collect()
assert t, f"target {TARGET_NAME} not found in BENCH_TARGETS"
t = t[0]
svc, pool, container = t["SERVICE_FQN"], t["CLIENT_POOL"], t["SERVER_CONTAINER"]

status = json.loads(session.sql("SELECT SYSTEM$GET_SERVICE_STATUS(?) AS S", params=[svc]).collect()[0]["S"])
print(f"Target service {svc}:")
for c in status:
    print(f"  {c.get('containerName'):<10} {c.get('status'):<10} {c.get('message','')}")
if not any(c.get("containerName") == container and c.get("status") == "READY" for c in status):
    print("  -> not READY yet; the run will RESUME it and wait (up to 45 min, model load ~10-15 min).")

if "coldstart" in TESTS or "replica" in TESTS:
    assert CONFIRM_DISRUPTIVE, "coldstart/replica restart or scale the service: set CONFIRM_DISRUPTIVE = True"
    assert t["ALLOW_DISRUPTIVE"], "this target does not allow disruptive tests (ALLOW_DISRUPTIVE = FALSE)"

assert re.fullmatch(r"[A-Za-z0-9_$\"]+", pool), "unexpected pool name"
p = session.sql(f"SHOW COMPUTE POOLS LIKE '{pool}'").collect()
if p:
    p = p[0]
    print(f"\nClient pool {pool}: state={p['state']} nodes={p['active_nodes']}/{p['max_nodes']} instance={p['instance_family']}")
others = [r for r in session.sql(f"SHOW SERVICES IN COMPUTE POOL {pool}").collect()
          if not r["name"].upper().startswith("BENCH_")]
if others:
    print("  WARNING: other services are using this pool and may block the bench job:")
    for r in others:
        print(f"    {r['database_name']}.{r['schema_name']}.{r['name']}  ({r['status']})")
else:
    print("  OK: no other services on this pool.")

busy = session.sql("SELECT RUN_ID, STATUS FROM BENCH_RUNS WHERE STATUS IN ('QUEUED','RUNNING','CANCEL_REQUESTED')").collect()
print("\nRuns already queued/running:", [(r["RUN_ID"], r["STATUS"]) for r in busy] or "none")

## 3. Submit a run

Inserts a QUEUED row into `BENCH_RUNS` (all values as bind variables) and triggers the task. **Every execution submits a new run**; a RUN_ID cannot be reused.

In [ ]:
exists = session.sql("SELECT COUNT(*) N FROM BENCH_RUNS WHERE RUN_ID = ?", params=[RUN_ID]).collect()[0]["N"]
assert exists == 0, f"RUN_ID {RUN_ID} already exists; change RUN_ID in section 1"

session.sql("""
    INSERT INTO BENCH_RUNS (RUN_ID, TARGET_NAME, TESTS, LEVELS, MIN_PROMPTS, SLA, CONFIRM_DISRUPTIVE, STATUS, NOTE, REQUESTED_BY)
    SELECT ?, ?, PARSE_JSON(?), ?, ?, PARSE_JSON(?), ?, 'QUEUED', ?, CURRENT_USER()
""", params=[RUN_ID, TARGET_NAME, json.dumps(TESTS), LEVELS, int(MIN_PROMPTS), json.dumps(SLA),
             bool(CONFIRM_DISRUPTIVE), NOTE]).collect()

try:
    session.sql("EXECUTE TASK BENCH_RUNNER").collect()
    print(f"Submitted {RUN_ID}; task triggered.")
except Exception as e:
    # Fails if the task is already running; the running instance picks up QUEUED rows.
    print(f"Submitted {RUN_ID}; task not triggered ({e}). If another run is in progress, it will be picked up next.")

## 4. Monitor progress

**Re-run this section as often as you like.** It polls for up to `MONITOR_MINUTES` and then exits; the run itself is not affected. It also shows the container status of the current bench job, so a job stuck in PENDING is visible.

In [ ]:
MONITOR_MINUTES = 20
POLL_SECONDS = 30

def run_state(rid):
    r = session.sql("SELECT STATUS, CURRENT_STEP, CURRENT_JOB, ERROR, STARTED_AT, FINISHED_AT FROM BENCH_RUNS WHERE RUN_ID = ?",
                    params=[rid]).collect()
    return r[0] if r else None

def job_state(job):
    if not job or not re.fullmatch(r"BENCH_[A-Z0-9_]+", job):
        return ""
    try:
        rows = session.sql(f"SHOW SERVICE CONTAINERS IN SERVICE {job}").collect()
        return "; ".join(f"{r['container_name']}={r['status']} {r['message'] or ''}".strip() for r in rows)
    except Exception:
        return "(job finished or dropped)"

seen = set()
deadline = time.time() + MONITOR_MINUTES * 60
while True:
    s = run_state(RUN_ID)
    if s is None:
        print("run not found"); break
    for e in session.sql("SELECT ts, test, event, detail FROM BENCH_RUN_LOG WHERE run_id = ? ORDER BY ts",
                         params=[RUN_ID]).collect():
        key = (str(e["TS"]), e["TEST"], e["EVENT"])
        if key not in seen:
            seen.add(key)
            print(f"{str(e['TS'])[:19]}  {e['TEST']:<18} {e['EVENT']:<22} {e['DETAIL'] or ''}")
    if s["STATUS"] in ("DONE", "DONE_WITH_ERRORS", "FAILED", "CANCELLED"):
        print(f"\n== {RUN_ID}: {s['STATUS']} ==", s["ERROR"] or "")
        break
    print(f"  ... {dt.datetime.utcnow():%H:%M:%S} UTC  status={s['STATUS']} step={s['CURRENT_STEP']}  job: {job_state(s['CURRENT_JOB'])}")
    if time.time() > deadline:
        print("\nMonitor timeout reached; the run continues in the background. Re-run this cell to keep watching.")
        break
    time.sleep(POLL_SECONDS)

### 4.1 Cancel a run (optional)

Set `DO_CANCEL = True` and run. The current bench job is dropped, and the orchestrator stops after the current step.

In [ ]:
DO_CANCEL = False
if DO_CANCEL:
    s = run_state(RUN_ID)
    session.sql("UPDATE BENCH_RUNS SET STATUS = 'CANCEL_REQUESTED' WHERE RUN_ID = ? AND STATUS IN ('QUEUED','RUNNING')",
                params=[RUN_ID]).collect()
    if s and s["CURRENT_JOB"] and re.fullmatch(r"BENCH_[A-Z0-9_]+", s["CURRENT_JOB"]):
        session.sql(f"DROP SERVICE IF EXISTS {s['CURRENT_JOB']}").collect()
    print("cancel requested:", RUN_ID)

## 5. Results and SLA verdicts

Reads the `BENCH_RESULTS` view for the runs in `COMPARE_RUNS` plus this `RUN_ID`. If a point was loaded more than once, only the latest row is kept.

In [ ]:
%%sql -r results
SELECT run_id, test, label, input_len, output_len, concurrency, num_prompts, failed,
       ROUND(ttft_p50_s, 3) AS ttft_p50_s, ROUND(ttft_p95_s, 3) AS ttft_p95_s,
       ROUND(tpot_p95_ms, 1) AS tpot_p95_ms, ROUND(per_req_tok_s_p95, 1) AS tok_s_p95,
       ROUND(e2e_p95_s, 2) AS e2e_p95_s, ROUND(out_tok_per_s, 1) AS out_tok_per_s
FROM {{BENCH_SCHEMA}}.BENCH_RESULTS
WHERE run_id IN ({{RUN_LIST_SQL}})
QUALIFY ROW_NUMBER() OVER (PARTITION BY run_id, test, label, input_len, concurrency ORDER BY loaded_at DESC) = 1
ORDER BY test, label, run_id, concurrency

In [ ]:
import pandas as pd
# Bounded: a few hundred rows at most (runs x tests x shapes x levels).
res = results.copy() if isinstance(results, pd.DataFrame) else results.limit(5000).to_pandas()
res.columns = [c.lower() for c in res.columns]

res["sla_ok"] = ((res.ttft_p95_s < SLA["ttft_p95_s"]) & (res.e2e_p95_s < SLA["e2e_p95_s"])
                 & (res.tok_s_p95 >= SLA["min_tok_s"]) & (res.failed == 0))
def why(r):
    f = []
    if r.ttft_p95_s >= SLA["ttft_p95_s"]: f.append("TTFT")
    if r.e2e_p95_s >= SLA["e2e_p95_s"]:   f.append("E2E")
    if r.tok_s_p95 < SLA["min_tok_s"]:    f.append("tok/s")
    if r.failed > 0:                      f.append("failures")
    return ",".join(f)
res["fails_on"] = res.apply(why, axis=1)

# Max concurrency that meets the SLA, per run and shape (all lower steps must also pass).
def max_ok(g):
    ok = 0
    for _, r in g.sort_values("concurrency").iterrows():
        if not r.sla_ok: break
        ok = r.concurrency
    return ok
summary = (res[res.test.isin(["ramp", "shape"])]
           .groupby(["test", "label", "run_id"]).apply(max_ok).rename("max_sla_concurrency").reset_index())
summary["single_req_tok_s"] = summary.apply(
    lambda s: res[(res.run_id == s.run_id) & (res.label == s.label) & (res.concurrency == 1)].tok_s_p95.max(), axis=1)
summary["peak_out_tok_s"] = summary.apply(
    lambda s: res[(res.run_id == s.run_id) & (res.label == s.label)].out_tok_per_s.max(), axis=1)
print("SLA:", SLA)
summary

### 5.1 Per-point detail (failing points show the reason)

In [ ]:
res[["run_id", "test", "label", "input_len", "output_len", "concurrency", "ttft_p95_s", "tok_s_p95",
     "e2e_p95_s", "out_tok_per_s", "failed", "sla_ok", "fails_on"]]

## 6. Charts: compare runs

Top row: concurrency ramp (per-request speed and TTFT with dashed SLA lines, plus total throughput). Bottom row: token shape, RAG vs chat.

In [ ]:
import matplotlib.pyplot as plt

def plot_set(df, title):
    if df.empty:
        print("no data for", title); return
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    for (rid, lab), g in df.groupby(["run_id", "label"]):
        g = g.sort_values("concurrency"); name = f"{rid} / {lab}"
        ax[0].plot(g.concurrency, g.tok_s_p95, marker="o", label=name)
        ax[1].plot(g.concurrency, g.ttft_p95_s, marker="o", label=name)
        ax[2].plot(g.concurrency, g.out_tok_per_s, marker="o", label=name)
    ax[0].axhline(SLA["min_tok_s"], ls="--", c="grey"); ax[0].set_title("Per-request tok/s (p95)")
    ax[1].axhline(SLA["ttft_p95_s"], ls="--", c="grey"); ax[1].set_title("TTFT p95 (s)"); ax[1].set_yscale("log")
    ax[2].set_title("Total output tok/s")
    for a in ax:
        a.set_xscale("log", base=2); a.set_xlabel("concurrency"); a.grid(alpha=.3)
    ax[2].legend(fontsize=8)
    fig.suptitle(title); plt.tight_layout(); plt.show()

plot_set(res[res.test == "ramp"], "Concurrency ramp")
plot_set(res[res.test == "shape"], "Token shape: RAG vs chat")

### 6.1 Baseline: effect of input length (concurrency 1)

In [ ]:
b = res[res.test == "baseline"].sort_values(["run_id", "input_len"])
if b.empty:
    print("no baseline data")
else:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for rid, g in b.groupby("run_id"):
        ax[0].plot(g.input_len, g.ttft_p95_s, marker="o", label=rid)
        ax[1].plot(g.input_len, g.tok_s_p95, marker="o", label=rid)
    ax[0].set_title("TTFT p95 (s) vs input length"); ax[1].set_title("Per-request tok/s vs input length")
    for a in ax:
        a.set_xscale("log", base=2); a.set_xlabel("input tokens"); a.grid(alpha=.3); a.legend()
    plt.tight_layout(); plt.show()

## 7. Cold start

Only populated when `coldstart` or `replica` was run. `restart` = suspend then resume the service; `scaleup` = seconds until the 2nd replica was READY.

In [ ]:
%%sql -r coldstart
SELECT run_id, mode, models_ready_s, first_response_s, resume_to_probe_s,
       ROUND((COALESCE(resume_to_probe_s, 0) + first_response_s) / 60, 1) AS total_min
FROM {{BENCH_SCHEMA}}.BENCH_COLDSTART
WHERE run_id IN ({{RUN_LIST_SQL}})
ORDER BY run_id, mode

## 8. Diagnostics: vLLM startup log

Pulls the key lines from the event table that explain performance:

- `non-default args`: the flags actually in effect, e.g. whether `enforce_eager` is on
- `Available KV cache memory` / `GPU KV cache size` / `Maximum concurrency`: KV cache size
- `Graph capturing finished`: CUDA graphs are enabled (absent in eager mode)
- `Loading weights took`: weight load time
- `No available memory` / `Error`: why startup failed

`LOOKBACK_HOURS` sets how far back to look; this scans the event table, so a wider window is slower.

In [ ]:
LOOKBACK_HOURS = 6
if not EVENT_TABLE:
    print("Set EVENT_TABLE in section 1 (find it with: SHOW PARAMETERS LIKE 'EVENT_TABLE' IN ACCOUNT)")
else:
    svc_name = svc.split(".")[-1].upper()
    assert re.fullmatch(r"[A-Z0-9_$]+", svc_name)
    pattern = (".*(non-default args|enforce_eager|Available KV cache|GPU KV cache size|Maximum concurrency|"
               "Graph capturing finished|Loading weights took|init engine|No available memory|Error).*")
    logs = session.sql(f"""
        SELECT TIMESTAMP AS ts, LEFT(VALUE::STRING, 400) AS line
        FROM {EVENT_TABLE}
        WHERE RESOURCE_ATTRIBUTES:"snow.service.name"::STRING = ?
          AND RESOURCE_ATTRIBUTES:"snow.service.container.name"::STRING = ?
          AND TIMESTAMP > DATEADD('hour', -?, CURRENT_TIMESTAMP())
          AND REGEXP_LIKE(VALUE::STRING, ?, 'is')
        ORDER BY TIMESTAMP DESC
        LIMIT 200
    """, params=[svc_name, container, int(LOOKBACK_HOURS), pattern]).collect()

    seen = set()
    for r in reversed(logs):
        msg = r["LINE"].split("]", 1)[-1].strip()
        if msg[:80] in seen:          # the same line is logged once per TP worker
            continue
        seen.add(msg[:80])
        print(str(r["TS"])[:19], msg[:260])
    print(f"\n{len(logs)} matching lines in the last {LOOKBACK_HOURS} h")

## 9. Clean up

The GPU pool bills while it is active. When the run has finished and nobody else needs the service, set `DO_SUSPEND = True` and run. Requires OPERATE on the service and pool.

In [ ]:
DO_SUSPEND = False
if DO_SUSPEND:
    s = run_state(RUN_ID)
    assert s is None or s["STATUS"] not in ("QUEUED", "RUNNING", "CANCEL_REQUESTED"), "a run is still active"
    session.sql(f"ALTER SERVICE {svc} SUSPEND").collect()
    print("suspended service", svc)
    # Suspend the GPU pool too if nothing else runs on it, e.g.:
    # session.sql("ALTER COMPUTE POOL <GPU_POOL> SUSPEND").collect()